# 07 - Bologna zonal analysis from post-processed shadow rasters

This notebook computes spatial statistics over Bologna polygon layers from the post-processed shadow rasters created in the previous step.

It generalizes the original monthly TOTAL-only script so that it can analyse:
- all detected `ombra_*.tif` rasters
- all supported products (`GROUND`, `FOOTPRINT`, `TOTAL`, `ROOF_SURFACE`)
- both monthly and global outputs

For each raster and each Bologna layer, the notebook exports:
- per-raster / per-layer CSV
- per-raster / per-layer GeoPackage
- grouped summaries
- cross-layer comparison tables
- a Markdown report
- an analysis manifest


## 1. Install the required libraries

Run this only if the current environment does not already contain the required dependencies.


In [ ]:
# %pip install rasterio geopandas pandas numpy requests


## 2. Import the libraries used by the zonal workflow

These imports cover:
- raster reading
- polygon masking
- Bologna Open Data download
- tabular summaries
- report generation


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import io
import json
import math
import re
import time
import warnings
from typing import Any

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.features import geometry_mask
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform_bounds


## 3. Configure the zonal analysis

Set:
- the folder containing the post-processed rasters
- the output folder
- the thresholds used for summary percentages
- optional filters on products or layers

By default, the notebook analyses all supported rasters and all configured Bologna layers.


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

@dataclass
class BolognaZonalConfig:
    raster_dir: str = "output_shadow_variants"
    output_dir: str = "bologna_shadow_zonal_analysis"

    api_base: str = "https://opendata.comune.bologna.it/api/explore/v2.1/catalog/datasets"

    bologna_layers: list[tuple[str, str, str, str | None]] = None

    bike_buffer_by_type: dict[str, float] = None
    bike_buffer_default: float = 1.5
    min_area_m2: float = 10.0
    thresholds: tuple[float, ...] = (0.25, 0.50, 0.75, 0.90)

    variant_filter: tuple[str, ...] | None = None
    layer_filter: tuple[str, ...] | None = None

    def __post_init__(self):
        if self.bologna_layers is None:
            self.bologna_layers = [
                ("aree-stradali", "streets", "polygon", "descrizion"),
                ("zona-pedonale-centro-storico", "pedestrian_zones", "polygon", None),
                ("un_gest", "green_areas", "polygon", "classe_unita_gestionale"),
                ("piste-ciclopedonali", "bike_lanes", "line", "dtipologia2"),
                ("aree-statistiche", "stat_zones", "polygon", "zona"),
                ("quartieri-di-bologna", "quartieri", "polygon", "quartiere"),
            ]
        if self.bike_buffer_by_type is None:
            self.bike_buffer_by_type = {
                "sede propria": 1.5,
                "corsia ciclabile su strada": 1.25,
                "ciclabile contigua al pedonale": 1.5,
                "promiscuo ciclopedonale": 2.0,
                "promiscuo veicolare": 1.75,
                "corsia preferenziale bus/bici": 2.0,
                "area pedonale": 3.0,
                "pavimentato": 1.5,
                "sterrato": 1.5,
            }

cfg = BolognaZonalConfig()
cfg


## 4. Parse shadow raster names and build a generalized raster manifest

This notebook recognizes post-processed rasters with names like:

- `ombra_202506_month_TOTAL.tif`
- `ombra_202506_earlymorning_GROUND.tif`
- `ombra_global_fullperiod_TOTAL.tif`
- `ombra_global_h06_ROOF_SURFACE.tif`

Each raster is parsed into:
- `scope_key` (`global` or `YYYYMM`)
- `temporal_key`
- `variant`
- `label`


In [ ]:
VARIANT_KEYS = ("GROUND", "FOOTPRINT", "TOTAL", "ROOF_SURFACE")

TEMPORAL_LABELS = {
    "month": "Monthly total",
    "fullperiod": "Full available period",
    "earlymorning": "Early morning (06:00–09:00)",
    "morning": "Morning (09:00–12:00)",
    "peakthermal": "Peak thermal (12:00–15:00)",
    "afternoon": "Afternoon (15:00–18:00)",
    "evening": "Evening (18:00–21:00)",
}


@dataclass(frozen=True)
class ProductSpec:
    filename: Path
    key: str
    scope_key: str
    temporal_key: str
    variant: str
    label: str


def temporal_label(key: str) -> str:
    if key in TEMPORAL_LABELS:
        return TEMPORAL_LABELS[key]
    if re.fullmatch(r"h\d{2}", key):
        return f"Hour {key[1:]}:00"
    return key


def discover_shadow_rasters(raster_dir: Path, variant_filter: tuple[str, ...] | None = None) -> list[ProductSpec]:
    specs: list[ProductSpec] = []
    rx = re.compile(r"^ombra_(global|\d{6})_([A-Za-z0-9]+)_(GROUND|FOOTPRINT|TOTAL|ROOF_SURFACE)\.tif$", flags=re.IGNORECASE)

    for tif in sorted(raster_dir.glob("ombra_*.tif")):
        m = rx.match(tif.name)
        if not m:
            continue

        scope_key = m.group(1)
        temporal_key = m.group(2).lower()
        variant = m.group(3).upper()

        if variant_filter and variant not in variant_filter:
            continue

        label = f"{scope_key} - {temporal_label(temporal_key)} - {variant}"
        specs.append(
            ProductSpec(
                filename=tif,
                key=f"{scope_key}_{temporal_key}_{variant}",
                scope_key=scope_key,
                temporal_key=temporal_key,
                variant=variant,
                label=label,
            )
        )

    return specs


specs = discover_shadow_rasters(Path(cfg.raster_dir).resolve(), cfg.variant_filter)
print(f"Shadow rasters found: {len(specs)}")
specs[:10]


## 5. Download and prepare Bologna Open Data layers

This section:
- downloads the configured vector layers from the Bologna Open Data API
- reprojects them to the raster CRS
- optionally buffers the bike-lane layer
- keeps only valid polygonal geometries for zonal analysis


In [ ]:
def download_bologna_dataset(api_base: str, dataset_id: str) -> gpd.GeoDataFrame:
    url = f"{api_base}/{dataset_id}/exports/geojson"
    resp = requests.get(url, timeout=180)
    resp.raise_for_status()
    return gpd.read_file(io.BytesIO(resp.content))


def buffer_bike_lanes(gdf: gpd.GeoDataFrame, bike_buffer_by_type: dict[str, float], bike_buffer_default: float) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    widths = []
    for _, row in gdf.iterrows():
        btype = str(row.get("dtipologia2", "")).lower().strip()
        widths.append(bike_buffer_by_type.get(btype, bike_buffer_default) * 2.0)
    gdf["lane_width_m"] = widths
    gdf["geometry"] = gdf.apply(lambda r: r.geometry.buffer(r["lane_width_m"] / 2.0), axis=1)
    return gdf


def load_bologna_layers(target_crs, cfg: BolognaZonalConfig) -> dict[str, gpd.GeoDataFrame]:
    print("Downloading Bologna Open Data layers...")
    out: dict[str, gpd.GeoDataFrame] = {}

    for dataset_id, layer_name, geom_type, group_col in cfg.bologna_layers:
        if cfg.layer_filter and layer_name not in cfg.layer_filter:
            continue

        print(f"  [{layer_name}] downloading {dataset_id}...")
        gdf = download_bologna_dataset(cfg.api_base, dataset_id)

        if gdf.empty:
            out[layer_name] = gdf
            continue

        if "fid" in gdf.columns:
            gdf = gdf.drop(columns=["fid"])

        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")

        gdf = gdf.to_crs(target_crs)

        if layer_name == "bike_lanes":
            gdf = buffer_bike_lanes(gdf, cfg.bike_buffer_by_type, cfg.bike_buffer_default)

        gdf = gdf[gdf.geometry.notnull()].copy()

        if geom_type == "polygon" or layer_name == "bike_lanes":
            gdf = gdf[gdf.geometry.area > cfg.min_area_m2].copy()

        gdf["__layer__"] = layer_name
        gdf["__group_col__"] = group_col if group_col else "__all__"
        out[layer_name] = gdf.reset_index(drop=True)

    return out


## 6. Prepare utility functions for raster access and polygon statistics

These helpers:
- transform raster bounds to WGS84 for reporting
- compute safe windows from geometry bounds
- mask pixels by geometry
- calculate continuous statistics such as mean, median, min, max, std and threshold percentages


In [ ]:
def safe_str(val: Any, max_len: int = 80) -> str:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "(N/A)"
    s = str(val)[:max_len]
    return s.encode("ascii", errors="replace").decode("ascii")


def get_feature_name(row) -> str:
    for col in [
        "nome", "quartiere", "zona", "descrizion", "classe_unita_gestionale",
        "dtipologia2", "tipoztl", "nomevia1", "nomevia2", "nomevia3", "street_name", "display_name"
    ]:
        if col in row.index:
            val = row.get(col)
            if val is not None and str(val) not in {"nan", "None", ""}:
                return safe_str(val)
    return "(unnamed)"


def get_raster_bounds_wgs84(tif_path: Path):
    with rasterio.open(tif_path) as src:
        b = src.bounds
        crs = src.crs
    west, south, east, north = transform_bounds(crs, "EPSG:4326", b.left, b.bottom, b.right, b.top)
    return (west, south, east, north), crs


def _safe_window(src: rasterio.io.DatasetReader, geom) -> Window | None:
    minx, miny, maxx, maxy = geom.bounds
    try:
        win = from_bounds(minx, miny, maxx, maxy, src.transform)
    except Exception:
        return None
    row_off = max(0, int(math.floor(win.row_off)))
    col_off = max(0, int(math.floor(win.col_off)))
    row_end = min(src.height, int(math.ceil(win.row_off + win.height)))
    col_end = min(src.width, int(math.ceil(win.col_off + win.width)))
    h = row_end - row_off
    w = col_end - col_off
    if h <= 0 or w <= 0:
        return None
    return Window(col_off, row_off, w, h)


def _geometry_pixel_mask(window_transform, out_shape, geom):
    return geometry_mask(
        [geom.__geo_interface__],
        transform=window_transform,
        invert=True,
        out_shape=out_shape,
        all_touched=True,
    )


def _continuous_stats(values: np.ndarray, thresholds: tuple[float, ...]) -> dict[str, Any]:
    if values.size == 0:
        return {
            "count": 0,
            "mean": np.nan,
            "median": np.nan,
            "min": np.nan,
            "max": np.nan,
            "std": np.nan,
            **{f"pct_ge_{int(t*100):02d}": np.nan for t in thresholds},
        }

    out = {
        "count": int(values.size),
        "mean": float(np.nanmean(values)),
        "median": float(np.nanmedian(values)),
        "min": float(np.nanmin(values)),
        "max": float(np.nanmax(values)),
        "std": float(np.nanstd(values)),
    }

    for t in thresholds:
        out[f"pct_ge_{int(t*100):02d}"] = float(np.mean(values >= t))

    return out


## 7. Compute feature-by-feature zonal statistics

For each polygon and each raster, the notebook:
- clips a safe read window
- masks pixels to the polygon geometry
- removes nodata / invalid pixels
- computes summary statistics from the valid values


In [ ]:
def zonal_stats_feature_by_feature(src: rasterio.io.DatasetReader, gdf: gpd.GeoDataFrame, thresholds: tuple[float, ...]) -> pd.DataFrame:
    rows = []
    nodata = src.nodata

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        window = _safe_window(src, geom)
        if window is None:
            continue

        arr = src.read(1, window=window, masked=False).astype(np.float32)
        w_transform = src.window_transform(window)
        gmask = _geometry_pixel_mask(w_transform, arr.shape, geom)

        valid = gmask.copy()

        if nodata is not None and not (isinstance(nodata, float) and np.isnan(nodata)):
            valid &= (arr != nodata)

        valid &= np.isfinite(arr)
        vals = arr[valid]

        stats = _continuous_stats(vals, thresholds)
        rec = {
            "feature_idx": int(idx),
            "feature_name": get_feature_name(row),
            "layer_name": row.get("__layer__", "unknown"),
            "group_col": row.get("__group_col__", "__all__"),
            "geometry_area_m2": float(geom.area) if geom is not None else np.nan,
        }

        for col in row.index:
            if col == "geometry":
                continue
            rec[col] = row[col]

        rec.update(stats)
        rows.append(rec)

    return pd.DataFrame(rows)


## 8. Report-writing helper

The workflow builds a Markdown report summarizing:
- which rasters were analysed
- per-layer grouped summaries
- global comparison tables


In [ ]:
def write_md_report(path: Path, title: str, blocks: list[str]):
    text = [f"# {title}", ""]
    text.extend(blocks)
    path.write_text("\n".join(text), encoding="utf-8")


## 9. Validate raster inputs and load Bologna layers

This block:
- discovers the generalized post-processed rasters
- reads the first raster to determine the working CRS
- downloads and prepares the Bologna layers


In [ ]:
specs = discover_shadow_rasters(Path(cfg.raster_dir).resolve(), cfg.variant_filter)
if not specs:
    raise FileNotFoundError(f"No supported shadow rasters found in {Path(cfg.raster_dir).resolve()}")

output_dir = Path(cfg.output_dir).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

raster_bounds_wgs84, target_crs = get_raster_bounds_wgs84(specs[0].filename)
layers = load_bologna_layers(target_crs, cfg)

print(f"Rasters to analyse: {len(specs)}")
print(f"Layers loaded: {list(layers.keys())}")
print(f"Output dir: {output_dir}")


## 10. Run the generalized Bologna zonal analysis

This is the main workflow.

For every Bologna layer and every shadow raster, the notebook exports:
- one CSV
- one GeoPackage
- grouped summaries by scope / temporal key / variant
- a global comparison table
- a Markdown report
- a manifest JSON


In [ ]:
all_rows = []
report_blocks: list[str] = []
t0 = time.time()

for layer_name, gdf in layers.items():
    if gdf.empty:
        print(f"Skipping empty layer: {layer_name}")
        continue

    print(f"\nProcessing layer: {layer_name} ({len(gdf)} features)")
    layer_out = output_dir / layer_name
    layer_out.mkdir(parents=True, exist_ok=True)
    layer_rows = []

    for spec in specs:
        print(f"  Raster: {spec.filename.name}")
        with rasterio.open(spec.filename) as src:
            df = zonal_stats_feature_by_feature(src, gdf, cfg.thresholds)

        if df.empty:
            continue

        df["raster_key"] = spec.key
        df["raster_label"] = spec.label
        df["scope_key"] = spec.scope_key
        df["temporal_key"] = spec.temporal_key
        df["temporal_label"] = temporal_label(spec.temporal_key)
        df["variant"] = spec.variant

        csv_path = layer_out / f"{spec.key}__{layer_name}.csv"
        df.to_csv(csv_path, index=False)

        gpkg_path = layer_out / f"{spec.key}__{layer_name}.gpkg"
        gdf_out = gdf.copy().loc[df["feature_idx"].astype(int).tolist()].reset_index(drop=True)
        stat_cols = [c for c in df.columns if c not in gdf_out.columns or c == "feature_idx"]
        df_merge = df[["feature_idx"] + [c for c in stat_cols if c != "feature_idx"]].copy()
        gdf_out["feature_idx"] = df["feature_idx"].astype(int).values
        gdf_out = gdf_out.merge(df_merge, on="feature_idx", how="left")
        gdf_out.to_file(gpkg_path, driver="GPKG")

        layer_rows.append(df)
        all_rows.append(df)

    if not layer_rows:
        continue

    layer_df = pd.concat(layer_rows, ignore_index=True)

    group_col = gdf["__group_col__"].iloc[0]
    group_keys = ["scope_key", "temporal_key", "variant"]
    if group_col != "__all__" and group_col in layer_df.columns:
        group_keys.append(group_col)

    grouped = (
        layer_df.groupby(group_keys, dropna=False)
        .agg(
            n_features=("feature_idx", "count"),
            mean_of_means=("mean", "mean"),
            median_of_medians=("median", "median"),
            min_mean=("mean", "min"),
            max_mean=("mean", "max"),
            avg_pct_ge_25=("pct_ge_25", "mean"),
            avg_pct_ge_50=("pct_ge_50", "mean"),
            avg_pct_ge_75=("pct_ge_75", "mean"),
            avg_pct_ge_90=("pct_ge_90", "mean"),
        )
        .reset_index()
        .sort_values(group_keys)
    )
    grouped.to_csv(layer_out / f"{layer_name}__grouped_summary.csv", index=False)

    report_blocks.append(f"## Layer: {layer_name}")
    report_blocks.append("")
    report_blocks.append(
        f"Features analysed: **{len(layer_df)}** rows across **{len(layer_rows)}** raster/layer combinations."
    )
    report_blocks.append("")
    layer_scope_temporal = (
        layer_df.groupby(["scope_key", "temporal_key", "variant"], dropna=False)
        .agg(
            mean_shadow=("mean", "mean"),
            median_shadow=("median", "median"),
            n=("feature_idx", "count"),
        )
        .reset_index()
        .sort_values(["scope_key", "temporal_key", "variant"])
    )
    report_blocks.append(layer_scope_temporal.to_markdown(index=False))
    report_blocks.append("")

if not all_rows:
    raise RuntimeError("No analysis results were produced.")

big_df = pd.concat(all_rows, ignore_index=True)
big_df.to_csv(output_dir / "all_layers_all_rasters_stats.csv", index=False)

comparison = (
    big_df.groupby(["layer_name", "scope_key", "temporal_key", "variant"], dropna=False)
    .agg(
        n_features=("feature_idx", "count"),
        mean_shadow=("mean", "mean"),
        median_shadow=("median", "median"),
        avg_pct_ge_50=("pct_ge_50", "mean"),
        avg_pct_ge_75=("pct_ge_75", "mean"),
        avg_pct_ge_90=("pct_ge_90", "mean"),
    )
    .reset_index()
    .sort_values(["scope_key", "temporal_key", "variant", "layer_name"])
)
comparison.to_csv(output_dir / "cross_layer_comparison.csv", index=False)

manifest = {
    "created_seconds": round(time.time() - t0, 2),
    "input_rasters": [str(s.filename.resolve()) for s in specs],
    "n_rasters": len(specs),
    "raster_bounds_wgs84": raster_bounds_wgs84,
    "output_dir": str(output_dir.resolve()),
    "layers": list(layers.keys()),
    "scope_keys": sorted({s.scope_key for s in specs}),
    "temporal_keys": sorted({s.temporal_key for s in specs}),
    "variants": sorted({s.variant for s in specs}),
}
(output_dir / "analysis_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

report_blocks.insert(0, f"Analysed rasters: **{len(specs)}** post-processed shadow rasters.")
report_blocks.insert(0, "")
report_blocks.insert(
    0,
    "This report summarizes Bologna zonal statistics for generalized post-processed shadow rasters."
)
write_md_report(output_dir / "bologna_shadow_analysis_report.md", "Bologna Shadow Zonal Analysis", report_blocks)

print(f"\nDone. Outputs written to: {output_dir.resolve()}")


## 11. Optional quick inspection of the main outputs

This is a simple post-run check that lists the main output files created by the notebook.


In [ ]:
print("Main outputs:")
for p in sorted(output_dir.glob("*")):
    print(" -", p.name)
